# 05d — Detector Comparison

Assembles the cross-framework benchmark from whichever of `05_yolo_fish_detection.ipynb`, `05b_rtdetr_fish_detection.ipynb`, and `05c_faster_rcnn_fish_detection.ipynb` have run and saved a metrics JSON (`aquatrack.benchmark.save_detector_metrics`). Does **not** retrain anything -- it loads each detector's already-saved weights fresh, benchmarks them identically (same validation images, same warmup/measurement split, same score/IoU operating point), and builds two tables:

1. **Overall**: precision, recall, mAP50, mAP50-95, F1, FPS, latency, parameter count, model size, peak GPU memory.
2. **Difficult-conditions breakdown**: recall specifically on validation images tagged small-fish / low-contrast / occluded / multi-fish / motion-blur / dark-scene -- because a detector's headline mAP can hide it quietly failing on exactly the frames this project cares about most.

**Do not pick the winner by mAP alone.** A slower, marginally more accurate model is not automatically the right choice for a C++ deployment target -- see the FPS/latency columns and the closing discussion cell.

**Kernel:** **AquaTrack+ (.venv)**.

## Final verified comparison — validation-selected checkpoints

This table uses the best validation checkpoint observed for each architecture. The earlier `rtdetr_l_5ep` run is intentionally excluded because it diverged to NaN. Epoch counts are recorded for transparency, not forced to match; Faster R-CNN's four-epoch run is preliminary because its original patience allowed only one epoch after the learning-rate reduction.

| Model | Epochs completed | Precision | Recall | F1 | mAP50 | mAP50–95 |
|---|---:|---:|---:|---:|---:|---:|
| YOLO11n | 13 (best 3) | 0.4190 | 0.4467 | 0.4324 | 0.3063 | 0.1354 |
| RT-DETR-L | 7 | **0.4322** | **0.5922** | **0.4997** | **0.3445** | **0.1446** |
| Faster R-CNN (ResNet50-FPN-v2) | 4 | 0.4237 | 0.3701 | 0.3951 | 0.2891 | 0.1144 |

**Result:** RT-DETR-L has the highest recorded mAP50–95, but YOLO11n is very close (0.1354 vs 0.1446) and has stronger qualitative predictions in the reviewed samples. Faster R-CNN ranks third. Precision/recall are shown for context but are not strictly cross-framework comparable because their operating-point selection differs.

![Final detector metric comparison](../runs/fish_detector_benchmark/detector_comparison_final.png)

Machine-readable table: [detector_comparison_final.csv](../runs/fish_detector_benchmark/detector_comparison_final.csv)

## Configuration

In [ ]:
# --- Edit for your run, then Run All ------------------------------------
DATASET_YAML = r"C:\Users\oyele\AquaTrack_data\sfishtrack_yolo\dataset.yaml"
METRICS_DIR = r"D:\Software_and_Web\softwares\AquaTrack+\runs\fish_detector_benchmark\metrics"
OUTPUT_DIR = r"D:\Software_and_Web\softwares\AquaTrack+\runs\fish_detector_benchmark"
DEVICE_INDEX = 0        # this machine's RTX 5070 Ti
SCORE_THRESHOLD = 0.5   # fixed operating point for precision/recall/F1 and the conditions table
IOU_THRESHOLD = 0.5     # match IoU for counting a prediction as correct
WARMUP = 10             # images run and discarded before timing starts
BENCHMARK = 100         # images actually timed, per model
MAX_IMAGES_PER_CONDITION = 150  # cap per-condition eval cost -- these subsets can otherwise run into the thousands
SEED = 42
# ------------------------------------------------------------------------

In [ ]:
import sys
sys.path.insert(0, r"D:\Software_and_Web\softwares\AquaTrack+\src")

from pathlib import Path
import pandas as pd
import yaml
from aquatrack.benchmark import load_detector_metrics

metrics_df = load_detector_metrics(METRICS_DIR)
# Compare one validation-selected result per architecture; exclude the failed NaN RT-DETR run.
selected_runs = {"sfishtrack_15ep", "rtdetr_l_converged", "faster_rcnn_converged"}
metrics_df = metrics_df[metrics_df["run_name"].isin(selected_runs)].copy()
assert not metrics_df.empty, (
    f"No metrics JSON found in {METRICS_DIR} -- run at least one of 05/05b/05c "
    "(through their 'save metrics' cell) before this notebook."
)
print(f"Found {len(metrics_df)} detector run(s):")
metrics_df[["run_name", "model", "framework", "epochs", "pilot"]]

In [ ]:
dataset_yaml_path = Path(DATASET_YAML)
assert dataset_yaml_path.is_file(), f"DATASET_YAML not found: {dataset_yaml_path}"
dataset_config = yaml.safe_load(dataset_yaml_path.read_text(encoding="utf-8"))
dataset_root = Path(dataset_config["path"])
num_classes = len(dataset_config["names"]) + 1  # +1 background, for reconstructing the Faster R-CNN skeleton

import torch
assert torch.cuda.is_available(), "CUDA not available -- check 00_environment_check.ipynb"
device = torch.device(f"cuda:{DEVICE_INDEX}")
print("GPU:", torch.cuda.get_device_name(device))

## A unified loader and predictor across three different frameworks

YOLO and RT-DETR share Ultralytics' `predict()` API; Faster R-CNN is called directly as a TorchVision `nn.Module`. Everything downstream (benchmarking, the conditions table) works off the common `(boxes_xyxy, scores)` this returns, so it doesn't need to know which framework it's looking at.

`conf=0.001` on the Ultralytics calls deliberately asks for almost every candidate box -- otherwise each framework's own default confidence threshold would silently pick a different operating point per model, and `SCORE_THRESHOLD` above stops being the actual comparison point.

In [ ]:
import cv2

def load_detector(row, device):
    framework = row["framework"]
    weights_path = row["weights_path"]
    if framework == "ultralytics_yolo":
        from ultralytics import YOLO
        return YOLO(weights_path)
    if framework == "ultralytics_rtdetr":
        from ultralytics import RTDETR
        return RTDETR(weights_path)
    if framework == "torchvision_fasterrcnn":
        from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
        # Reconstruct the same 640x640 transform used during training. Keeping
        # TorchVision's 800-1333 default here would make the speed comparison
        # inconsistent and reintroduce the large compute-time regression.
        model = fasterrcnn_resnet50_fpn_v2(
            weights=None, num_classes=num_classes, min_size=640, max_size=640,
        )
        model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
        return model.to(device).eval()
    raise ValueError(f"Unknown framework: {framework!r}")


def predict_boxes(framework, model, image_bgr, device):
    if framework in ("ultralytics_yolo", "ultralytics_rtdetr"):
        result = model.predict(image_bgr, device=device.index, conf=0.001, verbose=False)[0]
        return result.boxes.xyxy.cpu(), result.boxes.conf.cpu()
    if framework == "torchvision_fasterrcnn":
        rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        tensor = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255.0).unsqueeze(0).to(device)
        with torch.no_grad():
            prediction = model(tensor)[0]
        return prediction["boxes"].cpu(), prediction["scores"].cpu()
    raise ValueError(f"Unknown framework: {framework!r}")


def count_parameters(framework, model):
    module = model.model if framework in ("ultralytics_yolo", "ultralytics_rtdetr") else model
    return sum(p.numel() for p in module.parameters())


def load_gt_xyxy(image_path, labels_dir, width, height):
    label_path = labels_dir / f"{image_path.stem}.txt"
    boxes = []
    if label_path.is_file():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            _, cx, cy, w, h = (float(v) for v in line.split())
            boxes.append([(cx - w / 2) * width, (cy - h / 2) * height, (cx + w / 2) * width, (cy + h / 2) * height])
    return torch.tensor(boxes) if boxes else torch.zeros((0, 4))

## Overall comparison: quality + speed + footprint

Same fixed sample of validation images, same order, for every model -- loaded and benchmarked one at a time (not all three resident on the GPU together) so peak-memory readings aren't inflated by whichever model happened to load first.

In [ ]:
import random
import time
from aquatrack.benchmark import count_matches

labels_dir = dataset_root / "labels" / "val"
all_val_images = sorted((dataset_root / "images" / "val").glob("*.*"))
rng = random.Random(SEED)
benchmark_images = rng.sample(all_val_images, min(WARMUP + BENCHMARK, len(all_val_images)))

comparison_rows = []
for _, row in metrics_df.iterrows():
    print(f"Benchmarking {row['model']} ({row['run_name']})...")
    framework = row["framework"]
    model = load_detector(row, device)

    torch.cuda.reset_peak_memory_stats(device)
    latencies = []
    true_positives = false_positives = false_negatives = 0
    for i, image_path in enumerate(benchmark_images):
        image_bgr = cv2.imread(str(image_path))
        torch.cuda.synchronize()
        start = time.perf_counter()
        pred_boxes, pred_scores = predict_boxes(framework, model, image_bgr, device)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        if i >= WARMUP:
            latencies.append(elapsed)
            height, width = image_bgr.shape[:2]
            gt_boxes = load_gt_xyxy(image_path, labels_dir, width, height)
            tp, fp, fn = count_matches(
                pred_boxes, pred_scores, gt_boxes,
                score_threshold=SCORE_THRESHOLD, iou_threshold=IOU_THRESHOLD,
            )
            true_positives += tp; false_positives += fp; false_negatives += fn

    mean_latency = sum(latencies) / len(latencies)
    # Unlike the JSON headline values, these use the exact same confidence/IoU
    # protocol for every framework on the exact same benchmark images.
    precision = true_positives / (true_positives + false_positives) if true_positives + false_positives else 0.0
    recall = true_positives / (true_positives + false_negatives) if true_positives + false_negatives else 0.0
    comparison_rows.append({
        "model": row["model"],
        "precision": precision,
        "recall": recall,
        "map50": float(row["map50"]),
        "map50_95": float(row["map50_95"]),
        "f1": 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0,
        "fps": 1.0 / mean_latency,
        "latency_ms": mean_latency * 1000,
        "params_millions": count_parameters(framework, model) / 1e6,
        "model_size_mb": Path(row["weights_path"]).stat().st_size / 1e6,
        "peak_gpu_memory_mb": torch.cuda.max_memory_allocated(device) / 1e6,
        "epochs": int(row["epochs"]),
        "pilot": bool(row["pilot"]),
    })

    del model
    torch.cuda.empty_cache()

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

In [ ]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
comparison_path = output_dir / "detector_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print("Saved:", comparison_path)

if comparison_df["pilot"].any():
    print("\nNote: at least one run above is a pilot run -- do not draw conclusions "
          "about final model quality until every row is a validation-selected non-pilot run.")
else:
    print("\nEpoch counts are reported for transparency, not forced to match. Each final "
          "checkpoint should be selected by its own architecture's validation metric.")

## Tag validation images by difficulty

Thresholds are computed from this validation set's own distribution (25th/75th percentile), not guessed fixed numbers -- so they adapt to whatever this dataset actually looks like rather than assuming, say, a specific brightness value means "dark" regardless of the footage. Empty frames (no fish at all) are excluded from every tag -- an empty frame isn't "small fish" or "occluded", it's just empty.

In [ ]:
from torchvision.ops import box_iou
from aquatrack.video import frame_quality_metrics

image_records = []
for image_path in all_val_images:
    image_bgr = cv2.imread(str(image_path))
    height, width = image_bgr.shape[:2]
    gt_xyxy = load_gt_xyxy(image_path, labels_dir, width, height)
    quality = frame_quality_metrics(image_bgr)

    max_pairwise_iou = 0.0
    if gt_xyxy.shape[0] >= 2:
        ious = box_iou(gt_xyxy, gt_xyxy)
        ious.fill_diagonal_(0.0)
        max_pairwise_iou = float(ious.max())

    areas = ((gt_xyxy[:, 2] - gt_xyxy[:, 0]) * (gt_xyxy[:, 3] - gt_xyxy[:, 1])) / (width * height) if gt_xyxy.shape[0] else torch.zeros(0)
    image_records.append({
        "image_path": image_path,
        "gt_count": gt_xyxy.shape[0],
        "min_box_area": float(areas.min()) if areas.numel() else None,
        "brightness": quality["brightness"],
        "contrast": quality["contrast"],
        "sharpness": quality["sharpness"],
        "max_pairwise_iou": max_pairwise_iou,
    })
image_df = pd.DataFrame(image_records)

has_fish = image_df["gt_count"] > 0
small_fish_threshold = image_df.loc[has_fish, "min_box_area"].quantile(0.25)
dark_threshold = image_df.loc[has_fish, "brightness"].quantile(0.25)
low_contrast_threshold = image_df.loc[has_fish, "contrast"].quantile(0.25)
blur_threshold = image_df.loc[has_fish, "sharpness"].quantile(0.25)
multi_fish_threshold = image_df.loc[has_fish, "gt_count"].quantile(0.75)

image_df["small_fish"] = has_fish & (image_df["min_box_area"] <= small_fish_threshold)
image_df["low_contrast"] = has_fish & (image_df["contrast"] <= low_contrast_threshold)
image_df["occlusion"] = has_fish & (image_df["max_pairwise_iou"] > 0.1)
image_df["multiple_fish"] = has_fish & (image_df["gt_count"] >= multi_fish_threshold)
image_df["motion_blur"] = has_fish & (image_df["sharpness"] <= blur_threshold)
image_df["dark_scene"] = has_fish & (image_df["brightness"] <= dark_threshold)

CONDITIONS = ["small_fish", "low_contrast", "occlusion", "multiple_fish", "motion_blur", "dark_scene"]
for condition in CONDITIONS:
    print(f"{condition}: {int(image_df[condition].sum())} images")

## Recall per model, per difficult condition

In [ ]:
from aquatrack.benchmark import count_matches

condition_rows = []
for _, row in metrics_df.iterrows():
    framework = row["framework"]
    model = load_detector(row, device)
    print(f"Evaluating {row['model']} across difficult conditions...")

    for condition in CONDITIONS:
        subset = image_df[image_df[condition]]
        if len(subset) > MAX_IMAGES_PER_CONDITION:
            subset = subset.sample(MAX_IMAGES_PER_CONDITION, random_state=SEED)

        true_positives = false_positives = false_negatives = 0
        for _, image_row in subset.iterrows():
            image_path = image_row["image_path"]
            image_bgr = cv2.imread(str(image_path))
            height, width = image_bgr.shape[:2]
            gt_xyxy = load_gt_xyxy(image_path, labels_dir, width, height)
            pred_boxes, pred_scores = predict_boxes(framework, model, image_bgr, device)
            tp, fp, fn = count_matches(
                pred_boxes, pred_scores, gt_xyxy,
                score_threshold=SCORE_THRESHOLD, iou_threshold=IOU_THRESHOLD,
            )
            true_positives += tp; false_positives += fp; false_negatives += fn

        recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) else float("nan")
        condition_rows.append({"model": row["model"], "condition": condition, "n_images": len(subset), "recall": recall})

    del model
    torch.cuda.empty_cache()

condition_df = pd.DataFrame(condition_rows)
condition_table = condition_df.pivot(index="condition", columns="model", values="recall").reindex(CONDITIONS)
condition_table

In [ ]:
conditions_path = output_dir / "detector_comparison_conditions.csv"
condition_table.to_csv(conditions_path)
print("Saved:", conditions_path)

## Choosing a detector: not just "highest mAP"

The right pick balances detection accuracy, fish recall (especially on the difficult-conditions table above), inference speed, and C++ deployment practicality -- a model that's marginally more accurate but several times slower is often the wrong trade for a real-time application. Re-read `comparison_df` and `condition_table` together before deciding which detector(s) move on to `06_bytetrack_tracking.ipynb` / `07_botsort_tracking.ipynb`: either a single best-overall pick, or a best-speed and best-accuracy pair carried forward separately, per the project discussion that motivated this notebook.